# Dynamic Auto-Tagging with RAG-based Few-Shot Learning

This notebook demonstrates how to use dynamic prompting with TF-IDF retrieval for ticket classification.

## Setup & Initialization

In [12]:
import sys
import pandas as pd
import nest_asyncio

# 1. ALLOW ASYNC IN JUPYTER
nest_asyncio.apply()

# 2. FIX PATH TO PROJECT ROOT
sys.path.append('../../')

print(sys.path)

['/anaconda/envs/myvenv/lib/python311.zip', '/anaconda/envs/myvenv/lib/python3.11', '/anaconda/envs/myvenv/lib/python3.11/lib-dynload', '', '/anaconda/envs/myvenv/lib/python3.11/site-packages', '../../', '../../']


In [13]:
# Import the dynamic tagger
from utils.tagging.label_creation_dynamic_cisc import DynamicAutoTagger

print("Initializing Dynamic Tagger...")
# Initialize with config - using ../../ to reach project root
tagger = DynamicAutoTagger(
    config_path='../../config/config_labels_dynamic.yaml',
    # Silver Standard RAG database
    example_db_path='/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/labeled/lexicon_scenarios_5000.csv',  
    use_async=True 
)

print(f"✓ RAG Database loaded with {len(tagger.example_df)} examples.")
tagger.test_connection()

INFO:utils.tagging.label_creation_dynamic_cisc:Loading example database from /afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/labeled/lexicon_scenarios_5000.csv


Initializing Dynamic Tagger...


INFO:utils.tagging.label_creation_dynamic_cisc:Loaded 4982 labeled examples for RAG database.
INFO:utils.tagging.label_creation_dynamic_cisc:Initializing TF-IDF retriever...
INFO:utils.tagging.label_creation_dynamic_cisc:✓ TF-IDF retriever initialized


✓ RAG Database loaded with 4982 examples.


INFO:httpx:HTTP Request: POST https://ai-coe-openai-models.openai.azure.com/openai/deployments/gpt-5/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
INFO:utils.tagging.label_creation_dynamic_cisc:✓ Connection successful!


True

## Test the Retrieval System (Sanity Check)

In [14]:
# Test query
test_text = "Customer cannot login to YouSee Play app. Getting error message."
print(f"TEST QUERY: '{test_text}'\n")

# Retrieve similar examples
retrieved = tagger.retrieve_examples(test_text)

print(f"Retrieved {len(retrieved)} examples:")
print("-" * 50)
for i, ex in enumerate(retrieved, 1):
    print(f"Example {i} (Similarity: {ex['similarity']:.3f})")
    print(f"Text: {ex['text'][:120]}...")
    print(f"Label: {ex['label']}\n")

TEST QUERY: 'Customer cannot login to YouSee Play app. Getting error message.'

Retrieved 3 examples:
--------------------------------------------------
Example 1 (Similarity: 0.364)
Text: Support - My YouSee/Other The Customer has had too many login attempts and needs help unlocking their account. (YouSee P...
Label: (1g Self service, 2g YS-play), 3 Login issues

Example 2 (Similarity: 0.329)
Text: Support - TV/Other/Other When the Customer enters the YouSee Play app, it seems like the screen flickers, so the Custome...
Label: (1g Self service, 2g YS-play), 3 App

Example 3 (Similarity: 0.285)
Text: Support - Login/My YouSee/Dawn Native Customer (non-migrated)/It has worked before Customer cannot login into YouSee Mus...
Label: (1g Self service, 2g YouSee Music), 3 Login issues



## Load the Target Data

In [15]:
# Load your tickets to classify
df = pd.read_csv('/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/labeled/human_review_random_50.csv')
df_test = pd.read_csv('/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/labeled/silver_standard_dataset.csv')



In [16]:
# Create text column
#df['text'] = df['title'] + ' ' + df['description'] + ' ' + df['assignment_group']

# Select columns for prediction
prediction_df = df[["number", "text"]].copy()

print(f"Loaded {len(prediction_df)} tickets to classify.")
prediction_df.head(3)

Loaded 50 tickets to classify.


,number,text
0,INC0130953,Support - TV/Video on demand/Streamer or Audio...
1,INC0155292,Support - YouSee mail/Issue with Mit YouSee/Ca...
2,INC0122704,Support - My YouSee/App (Mobile/Tablet)/Missin...


## Run the Dynamic CISC Pipeline

In [17]:
# Run predictions with Dynamic Confidence-Informed Self-Consistency (CISC)
# - n_samples=10: Runs 10 independent reasoning paths per ticket
# - Output: A CSV with 'scientific_confidence', 'consistency', and dynamic 'shot' columns

#test_df = prediction_df.head(10).copy()

result_df = tagger.predict_and_create_csv(
    df=prediction_df,
    output_path='../../data/dynamic_predictions_cisc_test.csv',
    number_column='number',
    text_column='text',
    n_samples=5,        # <--- Request 10 samples for the CISC voting
    semaphore_limit=10   # Lower this if you hit Rate Limits (10 samples * 10 tickets = 100 concurrent calls)
)

print(f"\nTotal predictions: {len(result_df)}")
print(f"Unique tickets: {result_df['number'].nunique()}")

# Verify the new columns
# You should see: 'scientific_confidence', 'consistency', and the retrieved 'shot1', 'shot2', etc.
result_df.head(10)

Processing (Dynamic CISC, N=5):   0%|                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           | 0/50 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://ai-coe-openai-models.openai.azure.com/openai/deployments/gpt-5/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
Processing (Dynamic CISC, N=5):   6%|███████████████████████████████████████████████████████▊                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    


Total predictions: 50
Unique tickets: 50


,number,text,label,reasoning,scientific_confidence,reasoning_confidence,consistency,shot1,shot2,shot3
0,INC0146510,Support - My YouSee/Web (PC/Mac)/Missing conte...,"(1g Self service, 2g Mit YouSee), 3 Web",The issue is within Mit YouSee on web where th...,0.9994,1.00,1.0,Support - My YouSee/Web (PC/Mac)/Missing conte...,Support - My YouSee/Web (PC/Mac)/Missing conte...,Support - My YouSee/Web (PC/Mac)/Missing conte...
1,INC0136096,Support - Dawn/With customer context/Customer ...,"(1 Misc incidents, 2 Other), 3 CPR issues in Dawn",The issue is about inability to update a custo...,0.9994,1.00,1.0,Support - Dawn/With customer context/Customer ...,Support - Dawn/With Customer context/Customer ...,Support - Dawn/With Customer context/Customer ...
2,INC0130177,Support - YouSee mail/Issue within webmail / u...,"(1g Self service, 2g Webmail), 3 Fraud",The issue concerns YouSee Webmail with suspect...,0.9628,0.90,1.0,Support - YouSee mail/Issue within webmail / u...,Support - YouSee mail/Issue within webmail / u...,Support - YouSee mail/Issue within webmail / u...
3,INC0122704,Support - My YouSee/App (Mobile/Tablet)/Missin...,"(1g Self service, 2g YS-play), 3 Missing right...",The issue is with YouSee Play access rights (e...,0.9922,0.99,1.0,Support - My YouSee/App (Mobile/Tablet)/Missin...,Support - My YouSee/App (Mobile/Tablet)/Missin...,Support - TV/Other/Rights The Customer logs in...
4,INC0131120,Order is not received at vendor Summary:\nIt l...,"(1 Misc incidents, 2 Other), 3 Quote error",The issue is a generic order/quote fallout whe...,0.9988,1.00,1.0,[FO seq: 26300] Order is not received at vendo...,[FO seq: 53000]Quote occupied by itself Summar...,Quote is not completed Summary: \nError messa...
5,INC0133822,Support - Dawn/With Customer context/Customer ...,"(1 Misc incidents, 2 Other), 3 CPR issues in Dawn",The issue is that the customer’s address chang...,0.9976,1.00,1.0,Support - Dawn/With customer context/Customer ...,Support - Dawn/With Customer context/Customer ...,Support - Dawn/With Customer context/Customer ...
6,INC0153074,Support - Issues with Customer Data/Contact in...,"(1 Misc incidents, 2 Other), 3 Missing rights ...",The issue is about fields being greyed out and...,0.9274,0.95,1.0,Support - Issues with Customer Data/Contact in...,Support - Issues with Customer Data/Contact in...,Support - Issues with Customer Data/Contact in...
7,INC0148779,Support - Login/TV/Migrated Customer (born in ...,"(1g Self service, 2g YS-play), 3 Login issues",Customer cannot sign in to YouSee Play on TV v...,0.9910,0.98,1.0,Support - Login/TV/Migrated Customer (born in ...,Support - Login/My YouSee/Migrated Customer (b...,Support - Login/My YouSee/Migrated Customer (b...
8,INC0125987,Support - YouSee mail/Issue within webmail / u...,"(1g Self service, 2g Webmail), 3 technical issues",The description is about YouSee webmail where ...,0.9814,0.97,1.0,Support - YouSee mail/Issue within webmail / u...,Support - YouSee mail/Issue within webmail / u...,Support - YouSee mail/Issue within webmail / u...
9,INC0152173,Error with Upgrade Request and Downgrade Reque...,"(1 Misc incidents, 2 Other), 3 Tickets",The issue is a Dawn process failure where futu...,0.6160,0.72,0.6,Support - Dawn/With Customer context/Quotes an...,Support - Dawn/With Customer context/Quotes an...,Support - Dawn/With Customer context/Products/...


In [19]:
result_df.to_csv("/afh/projects/SY-Tagging-NOT_DELETE-7b113334-385d-4e41-87b7-c77611f91817/shared/Users/josta/Thesis_IT_TicketClassification/data/labeled/dynamic_tagging_results.csv", index=False)

In [20]:
# 1. Grab a sample ticket and its RAG examples
sample_text = prediction_df.iloc[49]['text']
retrieved = tagger.retrieve_examples(sample_text)
examples_text = tagger._format_examples(retrieved)

# 2. Fill the template with real data to measure size
full_prompt_sample = tagger.prompt_template.format(
    hierarchical_categories=tagger.hierarchical_categories,
    scenarios=tagger.scenarios,
    examples_text=examples_text,
    description=sample_text
)

# 3. Calculate metrics
char_count = len(full_prompt_sample)
est_tokens = char_count / 4  # Standard industry heuristic for English/Danish mix

print(f"--- Token Estimation for 1 Sample ---")
print(f"Total Characters: {char_count:,}")
print(f"Estimated Tokens: {est_tokens:,.0f}")
print(f"Estimated Cost (Input): ${(est_tokens / 1_000_000) * 1.25:.4f} per sample")
print(f"Total Estimated Cost for 25,000 tickets (5 samples each): ${(est_tokens * 25000 * 5 / 1_000_000) * 1.25:,.2f}")

--- Token Estimation for 1 Sample ---
Total Characters: 5,679
Estimated Tokens: 1,420
Estimated Cost (Input): $0.0018 per sample
Total Estimated Cost for 25,000 tickets (5 samples each): $221.84


In [21]:
import asyncio

async def debug_llm_output():
    # 1. Use the same test ticket from your previous check
    test_text = prediction_df.iloc[0]['text']
    
    # 2. Get the RAG examples
    retrieved = tagger.retrieve_examples(test_text)
    examples_text = tagger._format_examples(retrieved)
    
    # 3. Request just ONE raw sample (no parsing yet)
    print("Requesting raw response from Azure OpenAI...")
    raw_responses = await tagger.predict_samples_async(
        description=test_text,
        examples_text=examples_text,
        n=1,
        temperature=1.0
    )
    
    print("\n--- RAW AI RESPONSE (THE 'TRUTH') ---")
    print(raw_responses[0])
    print("-------------------------------------")

# Run it in Jupyter
await debug_llm_output()

Requesting raw response from Azure OpenAI...


INFO:httpx:HTTP Request: POST https://ai-coe-openai-models.openai.azure.com/openai/deployments/gpt-5/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"



--- RAW AI RESPONSE (THE 'TRUTH') ---
(1c TV, 2c OTT), 3 Performance issues
Reasoning: The issue is lagging programs on a YouSee Play streamer, which points to OTT TV service performance degradation on the streaming device.
Confidence: 0.92
-------------------------------------


## Examine & Analyze Results

In [22]:
# Load results
results_df = pd.read_csv('../../data//labeled/dynamic_tagging_results.csv')

print(f"Total rows predicted: {len(results_df)}")
print("-" * 50)

# Pick the first ticket to examine deeply
first_ticket = results_df.iloc[0]

print(f"TICKET: {first_ticket['number']}")
print(f"TEXT: {first_ticket['text'][:150]}...\n")

print(f"FINAL PREDICTION:")
print(f"  Label: {first_ticket['label']}")
print(f"  Scientific Confidence (S*): {first_ticket['scientific_confidence']}")
print(f"  Consistency (Voting Agreement): {first_ticket['consistency']}\n")

print(f"AI REASONING:")
print(f"  {first_ticket['reasoning']}\n")

print(f"DYNAMIC EXAMPLES USED (RAG Context):")
print(f"  Shot 1: {first_ticket.get('shot1', 'N/A')}")
print(f"  Shot 2: {first_ticket.get('shot2', 'N/A')}")
print(f"  Shot 3: {first_ticket.get('shot3', 'N/A')}")

Total rows predicted: 50
--------------------------------------------------
TICKET: INC0146510
TEXT: Support - My YouSee/Web (PC/Mac)/Missing content (e.g. Bills) missing pay now button, and Customer is logged in with my ID.

Please look into why the ...

FINAL PREDICTION:
  Label: (1g Self service, 2g Mit YouSee), 3 Web
  Scientific Confidence (S*): 0.9994
  Consistency (Voting Agreement): 1.0

AI REASONING:
  The issue is within Mit YouSee on web where the “Pay now” button is missing after successful login. This aligns with Self service > Mit YouSee and a web-related scenario.

DYNAMIC EXAMPLES USED (RAG Context):
  Shot 1: Support - My YouSee/Web (PC/Mac)/Missing content (e.g. Bills) Customer has issues with their payment button being missing, and therefore they cannot pay their bill.

Category:Mit YouSee/Web (PC/Mac)/Missing content (e.g., Bills)
errorURL:https://wp-host-app-prod-bss.prod01.nc.nuuday.nu/customer/ticket-creation?customerId=309ca384-e2a1-4cea-85a8-e3ee3847d49b&showPl

## 8. Run on Full Dataset

In [ ]:
# Uncomment to run on full dataset
# results_full = tagger.predict_and_create_csv(
#     df=prediction_df,
#     output_path='../data/dynamic_predictions_full.csv',
#     number_column='number',
#     text_column='text'
# )

## 9. Compare Static vs Dynamic Prompting

In [ ]:
# Load static predictions (if you have them)
# static_results = pd.read_csv('../data/prediction_results.csv')
# dynamic_results = pd.read_csv('../data/dynamic_predictions_full.csv')

# # Compare top1 accuracy or other metrics
# # Add your comparison analysis here

## Key Differences from Static Prompting:

### Static Prompting:
- Uses same 7 hardcoded examples for ALL tickets
- Examples may not be relevant to the input

### Dynamic Prompting (This Notebook):
- Retrieves 3 most similar examples for EACH ticket
- Examples are always relevant to the input
- Stores which examples were used (shot1, shot2, shot3)
- Can analyze which examples lead to better predictions

### Output Format:
```
number | text | label | reasoning | confidence_score | shot1 | shot2 | shot3
```

Each ticket gets 3 rows (top3 predictions), all with the same retrieved examples.